# Post-Processing Classical Pipeline

Input: anomaly_scores.csv  
Output: classical_anomaly_report.csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import config as cfg
import warnings
warnings.filterwarnings("ignore")

## 1. Load Anomaly Scores

In [2]:
ANOMALY_DIR = cfg.IO_DIR / "anomaly_detection"
df = pd.read_csv(ANOMALY_DIR / "anomaly_scores.csv")
df["date"] = pd.to_datetime(df["date"])

Same as before, we track the time

In [3]:
import time
pipeline_start = time.time()

## 2. Business Rules

Not all statistical anomalies are operationally meaningful, so we apply four domain-specific rules on top of the model output. We flag observations where the flag rate exceeds 3× its recent rolling average, where passenger volume exceeds 3× baseline, where entries deviate by more than 2σ from the monthly norm, or where the alarm density is above the 95th percentile. An observation that triggers any of these rules is marked as rule-flagged.

In [4]:
df["rule_flag_spike"] = (df["flag_rate_dev_ratio7"] > 3).astype(int)
df["rule_entries_spike"] = (df["entries_dev_ratio7"] > 3).astype(int)
df["rule_high_residual_z"] = (df["entries_residual_z"].abs() > 2).astype(int)

alarm_95 = df["alarm_density_per_entry"].quantile(0.95)
df["rule_high_alarm"] = (df["alarm_density_per_entry"] > alarm_95).astype(int)

df["rule_any"] = df[["rule_flag_spike", "rule_entries_spike",
                      "rule_high_residual_z", "rule_high_alarm"]].max(axis=1)
print(f"Rules triggered: {df['rule_any'].sum()} / {len(df)}")

Rules triggered: 430 / 3449


## 3. Final Anomaly Label

The final anomaly label combines model consensus and business rules:

final_anomaly = consensus_anomaly OR rule_any

This ensures that:
Statistically unusual observations are captured even if no single business rule fires
Operationally meaningful patterns are captured even if the statistical models don't flag them

In [5]:
df["final_anomaly"] = ((df["consensus_anomaly"] == 1) | (df["rule_any"] == 1)).astype(int)
print(f"Final anomalies: {df['final_anomaly'].sum()} / {len(df)} ({df['final_anomaly'].mean():.1%})")

Final anomalies: 487 / 3449 (14.1%)


In [6]:
pipeline_end = time.time()
print(f"Classical pipeline execution time: {pipeline_end - pipeline_start:.2f} seconds")

Classical pipeline execution time: 0.11 seconds


## File Export

In [7]:
ANOMALY_DIR = cfg.IO_DIR / "anomaly_detection"
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)
output_path = ANOMALY_DIR / "classical_post_processed.csv"
df.to_csv(output_path, index=False)